In [8]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, FewShotPromptTemplate, PromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model="gpt-5.6-terra",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

examples = [
    {"movie": "탑건", "answer": "🛩️👨‍✈️🔥"},
    {"movie": "대부", "answer": "👨‍👨‍👦🔫🍝"},
    {"movie": "타이타닉", "answer": "🚢💑🧊"},
]

example_prompt = PromptTemplate.from_template("영화: {movie}\n이모지: {answer}")

fewshot_prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="",
)

examples_text = fewshot_prompt.format()

memory = ConversationBufferMemory(return_messages=True)

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 영화를 이모지로 표현하는 전문가입니다.
영화 제목을 받으면 그 영화를 나타내는 이모지 3개로만 답하고, 다른 글자는 쓰지 마세요.
영화 제목이 아닌 질문을 받으면 대화 기록을 참고해 한국어 문장으로 답하세요.

답변 형식 예시:
{examples}"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
]).partial(examples=examples_text)


def load_memory(_):
    return memory.load_memory_variables({})["history"]


chain = RunnablePassthrough.assign(history=load_memory) | prompt | chat


def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context({"input": question}, {"output": result.content})


invoke_chain("기생충")
invoke_chain("인터스텔라")

🏠🪨🪳🚀🌌⏳

In [9]:
invoke_chain("먼저 질문한 영화가 뭐였지?")

먼저 질문한 영화는 기생충이었어요.